## Run Guide (When to Use Which Command)
Use this section as the quick operating guide for local development and Slurm runs.

### 1) Before anything: run setup cells in order
Run the notebook top-to-bottom at least once so all functions/classes are defined.

### 2) Preflight only (no training, safest)
Use this when you are unsure whether train/val/test images are staged correctly.

```python
pipeline_results = run_full_pipeline_all_datasets(run_training=False)
pipeline_results
```

Expected behavior: reports per-dataset image availability like `found/checked`.
If `0/64`, the code is fine but the corresponding image files are not staged yet.

### 3) Smoke test (short debug run)
Use this to validate the training pipeline quickly before spending cluster hours.

```python
pipeline_results = run_full_pipeline_all_datasets(
    run_training=True,
    epochs=1,
    max_train_batches=10,
    max_val_batches=5,
    max_export_batches=5,
    export_split="test",
)
```

This verifies: dataloaders, forward/backward pass, checkpoint writes, softmax export.

### 4) Full training for one dataset
Use this when you want to train a single model end-to-end and export test softmax.

```python
history = fit_dataset_model("octmnist", epochs=20)
softmax_path = export_softmax_predictions(
    models_by_dataset["octmnist"],
    "octmnist",
    split="test",
)
```

### 5) Full training for all three datasets
Use this for full production run after data staging is confirmed.

```python
pipeline_results = run_full_pipeline_all_datasets(
    run_training=True,
    epochs=20,
    export_split="test",
)
pipeline_results
```

Outputs created:
- Checkpoints: `CNN/checkpoints/<dataset>/best.pt` and `CNN/checkpoints/<dataset>/last.pt`
- Softmax CSVs: `CNN/outputs/<dataset>_test_softmax.csv`

### 6) Slurm usage pattern
Use this when running on HPC where notebooks are executed non-interactively.

1. Activate environment in job script:
   - `source /home/verdantliberatus/University/CMPUT469/repository/Snowdrop469Project/.venv/bin/activate`
2. Execute notebook from command line (example):
   - `jupyter nbconvert --to notebook --execute CNN/train.ipynb --output train.executed.ipynb`
3. For first cluster run, prefer smoke settings (Section 3), then switch to full settings (Section 5).

### 7) Common failure modes
- **`No train images found ...`**: data files are not staged at expected paths under `datasets/images/<dataset>/...`
- **CPU instead of GPU**: check Slurm resource request and CUDA availability on allocated node


In [24]:
from pathlib import Path
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


@dataclass
class TrainConfig:
    seed: int = 42
    image_size: int = 28
    batch_size: int = 128
    num_workers: int = 4
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 20


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


config = TrainConfig()
set_seed(config.seed)

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = PROJECT_ROOT / "datasets"
IMAGE_ROOT = DATA_DIR / "images"

DATASET_NAMES = ["octmnist", "pathmnist", "tissuemnist"]
USE_SPLITS = ["train", "val", "test"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")

Device: cpu
Project root: /home/verdantliberatus/University/CMPUT469/repository/Snowdrop469Project
Data dir: /home/verdantliberatus/University/CMPUT469/repository/Snowdrop469Project/datasets


In [25]:
def load_split_labels(dataset_name: str, split: str) -> pd.DataFrame:
    csv_path = DATA_DIR / f"{dataset_name}_{split}_labels.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"image_index", "label_index", "label_name", "split"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path.name} missing columns: {missing}")
    return df


def infer_num_classes(dataset_name: str) -> int:
    train_df = load_split_labels(dataset_name, "train")
    return int(train_df["label_index"].nunique())


def resolve_image_path(dataset_name: str, split: str, image_index: int) -> Path:
    candidate_paths = [
        IMAGE_ROOT / dataset_name / split / f"{image_index}.png",
        IMAGE_ROOT / dataset_name / f"{image_index}.png",
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    dataset_dir = IMAGE_ROOT / dataset_name
    if dataset_dir.exists():
        wildcard_hits = list(dataset_dir.glob(f"**/{image_index}.png"))
        if wildcard_hits:
            return wildcard_hits[0]

    raise FileNotFoundError(
        f"Could not find image {image_index}.png under {IMAGE_ROOT / dataset_name}"
    )


class MedMNISTCsvDataset(Dataset):
    def __init__(self, dataset_name: str, split: str, transform=None):
        self.dataset_name = dataset_name
        self.split = split
        self.labels_df = load_split_labels(dataset_name, split)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.labels_df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        row = self.labels_df.iloc[idx]
        image_path = resolve_image_path(
            dataset_name=self.dataset_name,
            split=self.split,
            image_index=int(row["image_index"]),
        )

        image = Image.open(image_path).convert("RGB")
        label = int(row["label_index"])

        if self.transform is not None:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        return image, label


train_transform = transforms.Compose([
    transforms.Resize((config.image_size, config.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((config.image_size, config.image_size)),
    transforms.ToTensor(),
])

In [26]:
# Quick sanity check: CSV stats + currently available image files
for dataset_name in DATASET_NAMES:
    train_df = load_split_labels(dataset_name, "train")
    val_df = load_split_labels(dataset_name, "val")
    test_df = load_split_labels(dataset_name, "test")

    num_classes = infer_num_classes(dataset_name)
    print(f"\n[{dataset_name}] classes={num_classes} | train={len(train_df)} | val={len(val_df)} | test={len(test_df)}")

    sample_indices = train_df["image_index"].head(32).tolist()
    found = 0
    for image_index in sample_indices:
        try:
            _ = resolve_image_path(dataset_name, "train", int(image_index))
            found += 1
        except FileNotFoundError:
            pass
    print(f"  sample availability (first 32 train rows): {found}/32")


[octmnist] classes=4 | train=87835 | val=9761 | test=901
  sample availability (first 32 train rows): 32/32

[pathmnist] classes=9 | train=80917 | val=8995 | test=6455
  sample availability (first 32 train rows): 32/32

[tissuemnist] classes=8 | train=157896 | val=22557 | test=45121
  sample availability (first 32 train rows): 32/32


In [27]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )
        feature_size = config.image_size // 8
        flattened_dim = 128 * feature_size * feature_size
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        logits = self.classifier(x)
        return logits


def build_model_for_dataset(dataset_name: str) -> nn.Module:
    num_classes = infer_num_classes(dataset_name)
    model = SimpleCNN(num_classes=num_classes).to(device)
    return model


@torch.no_grad()
def predict_softmax(model: nn.Module, images: torch.Tensor) -> torch.Tensor:
    model.eval()
    logits = model(images.to(device))
    probs = torch.softmax(logits, dim=1)
    return probs


models_by_dataset = {name: build_model_for_dataset(name) for name in DATASET_NAMES}
for name, model in models_by_dataset.items():
    n_classes = infer_num_classes(name)
    print(f"[{name}] model ready | output classes={n_classes}")

[octmnist] model ready | output classes=4
[pathmnist] model ready | output classes=9
[tissuemnist] model ready | output classes=8


In [28]:
# Quick shape sanity check for softmax outputs
dummy_batch = torch.randn(4, 3, config.image_size, config.image_size, device=device)
for name, model in models_by_dataset.items():
    probs = predict_softmax(model, dummy_batch)
    row_sums = probs.sum(dim=1)
    print(f"[{name}] probs shape={tuple(probs.shape)} | row_sums={row_sums.cpu().numpy()}")

[octmnist] probs shape=(4, 4) | row_sums=[1.         0.99999994 1.         1.        ]
[pathmnist] probs shape=(4, 9) | row_sums=[1.0000001 1.        1.        1.       ]
[tissuemnist] probs shape=(4, 8) | row_sums=[1.         0.99999994 0.99999994 1.0000001 ]


In [29]:
@torch.no_grad()
def export_softmax_predictions(
    model: nn.Module,
    dataset_name: str,
    split: str = "test",
    output_dir: Optional[Path] = None,
    max_batches: Optional[int] = None,
    ) -> Path:
    ds = MedMNISTCsvDataset(dataset_name, split, transform=eval_transform)
    loader = DataLoader(
        ds,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    model.eval()
    all_probs = []
    all_labels = []
    image_indices = []
    offset = 0

    for batch_idx, (images, labels) in enumerate(loader):
        probs = predict_softmax(model, images).cpu().numpy()
        labels_np = labels.cpu().numpy()

        batch_size = labels_np.shape[0]
        batch_indices = ds.labels_df.iloc[offset:offset + batch_size]["image_index"].astype(int).tolist()
        offset += batch_size

        all_probs.append(probs)
        all_labels.extend(labels_np.tolist())
        image_indices.extend(batch_indices)

        if max_batches is not None and (batch_idx + 1) >= max_batches:
            break

    if not all_probs:
        raise RuntimeError("No predictions were generated. Check split/image availability.")

    probs_arr = np.vstack(all_probs)
    num_classes = probs_arr.shape[1]
    prob_cols = [f"prob_class_{i}" for i in range(num_classes)]

    out_df = pd.DataFrame(probs_arr, columns=prob_cols)
    out_df.insert(0, "true_label", all_labels)
    out_df.insert(0, "image_index", image_indices)

    resolved_output_dir = output_dir or globals().get("OUTPUT_DIR", PROJECT_ROOT / "CNN" / "outputs")
    resolved_output_dir.mkdir(parents=True, exist_ok=True)
    out_path = resolved_output_dir / f"{dataset_name}_{split}_softmax.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Saved softmax predictions: {out_path} | rows={len(out_df)}")
    return out_path


# Example usage when images are staged:
# history = fit_dataset_model("octmnist", epochs=20)
# export_softmax_predictions(models_by_dataset["octmnist"], "octmnist", split="test")

In [30]:
CHECKPOINT_DIR = PROJECT_ROOT / "CNN" / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "CNN" / "outputs"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def preflight_image_availability(dataset_name: str, split: str = "train", sample_size: int = 64) -> Tuple[int, int]:
    df = load_split_labels(dataset_name, split)
    sample_indices = df["image_index"].head(sample_size).tolist()
    found = 0
    for image_index in sample_indices:
        try:
            _ = resolve_image_path(dataset_name, split, int(image_index))
            found += 1
        except FileNotFoundError:
            pass
    return found, len(sample_indices)


def make_dataloaders(dataset_name: str) -> Dict[str, DataLoader]:
    train_ds = MedMNISTCsvDataset(dataset_name, "train", transform=train_transform)
    val_ds = MedMNISTCsvDataset(dataset_name, "val", transform=eval_transform)
    test_ds = MedMNISTCsvDataset(dataset_name, "test", transform=eval_transform)

    loaders = {
        "train": DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers, pin_memory=torch.cuda.is_available()),
        "val": DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers, pin_memory=torch.cuda.is_available()),
        "test": DataLoader(test_ds, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers, pin_memory=torch.cuda.is_available()),
    }
    return loaders


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: Optional[torch.optim.Optimizer],
    train_mode: bool,
    max_batches: Optional[int] = None,
    dataset_name: Optional[str] = None,
    split_name: Optional[str] = None,
    epoch_idx: Optional[int] = None,
    total_epochs: Optional[int] = None,
    print_every: int = 50,
    ) -> Dict[str, float]:
    model.train(train_mode)
    running_loss = 0.0
    total = 0
    correct = 0

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        if train_mode:
            loss.backward()
            optimizer.step()

        preds = torch.argmax(logits, dim=1)
        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size
        correct += (preds == labels).sum().item()

        if batch_idx % print_every == 0:
            split_label = split_name or ("train" if train_mode else "eval")
            prefix = f"[{dataset_name}] " if dataset_name else ""
            epoch_part = f"Epoch {epoch_idx}/{total_epochs} | " if epoch_idx is not None and total_epochs is not None else ""
            print(
                f"{prefix}{epoch_part}{split_label} batch {batch_idx}: "
                f"loss={loss.item():.4f} acc={(preds == labels).float().mean().item():.4f}"
            )

        if max_batches is not None and (batch_idx + 1) >= max_batches:
            break

    if total == 0:
        raise RuntimeError("No samples were processed in this epoch. Check dataset/image paths.")

    avg_loss = running_loss / total
    avg_acc = correct / total
    return {"loss": avg_loss, "acc": avg_acc}


def fit_dataset_model(
    dataset_name: str,
    epochs: Optional[int] = None,
    max_train_batches: Optional[int] = None,
    max_val_batches: Optional[int] = None,
) -> Dict[str, List[float]]:
    found, checked = preflight_image_availability(dataset_name, split="train", sample_size=64)
    print(f"[{dataset_name}] preflight train image availability: {found}/{checked}")
    if found == 0:
        raise RuntimeError(
            f"No train images found for {dataset_name}. Stage train/val/test images under {IMAGE_ROOT / dataset_name} before training."
        )

    loaders = make_dataloaders(dataset_name)
    model = build_model_for_dataset(dataset_name)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)

    total_epochs = epochs or config.epochs
    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [],
    }

    best_val_acc = -1.0
    ckpt_dir = CHECKPOINT_DIR / dataset_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_path = ckpt_dir / "best.pt"
    last_path = ckpt_dir / "last.pt"

    for epoch in range(1, total_epochs + 1):
        train_metrics = run_epoch(
            model=model,
            loader=loaders["train"],
            criterion=criterion,
            optimizer=optimizer,
            train_mode=True,
            max_batches=max_train_batches,
            dataset_name=dataset_name,
            split_name="train",
            epoch_idx=epoch,
            total_epochs=total_epochs,
        )
        val_metrics = run_epoch(
            model=model,
            loader=loaders["val"],
            criterion=criterion,
            optimizer=None,
            train_mode=False,
            max_batches=max_val_batches,
            dataset_name=dataset_name,
            split_name="val",
            epoch_idx=epoch,
            total_epochs=total_epochs,
        )

        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["acc"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_acc"].append(val_metrics["acc"])

        print(
            f"[{dataset_name}] epoch {epoch}/{total_epochs} | "
            f"train_loss={train_metrics['loss']:.4f} train_acc={train_metrics['acc']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f}"
        )

        checkpoint = {
            "dataset_name": dataset_name,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "config": vars(config),
        }
        torch.save(checkpoint, last_path)

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            torch.save(checkpoint, best_path)
            print(f"[{dataset_name}] saved new best checkpoint -> {best_path}")

    models_by_dataset[dataset_name] = model
    return history

In [31]:
# Preflight-only check (safe to run even with partial image staging)
for dataset_name in DATASET_NAMES:
    found, checked = preflight_image_availability(dataset_name, split="train", sample_size=64)
    print(f"[{dataset_name}] preflight train availability: {found}/{checked}")

[octmnist] preflight train availability: 64/64
[pathmnist] preflight train availability: 64/64
[tissuemnist] preflight train availability: 64/64


In [32]:
def run_full_pipeline_all_datasets(
    dataset_names: Optional[List[str]] = None,
    epochs: Optional[int] = None,
    run_training: bool = False,
    export_split: str = "test",
    max_train_batches: Optional[int] = None,
    max_val_batches: Optional[int] = None,
    max_export_batches: Optional[int] = None,
) -> Dict[str, Dict[str, object]]:
    selected = dataset_names or DATASET_NAMES
    results: Dict[str, Dict[str, object]] = {}

    for dataset_name in selected:
        print(f"\n=== {dataset_name} ===")
        found, checked = preflight_image_availability(dataset_name, split="train", sample_size=64)
        print(f"preflight train availability: {found}/{checked}")

        dataset_result: Dict[str, object] = {
            "preflight_found": found,
            "preflight_checked": checked,
            "trained": False,
            "exported": False,
        }

        if not run_training:
            dataset_result["message"] = "Preflight only. Set run_training=True to train/export."
            results[dataset_name] = dataset_result
            continue

        history = fit_dataset_model(
            dataset_name=dataset_name,
            epochs=epochs,
            max_train_batches=max_train_batches,
            max_val_batches=max_val_batches,
        )
        dataset_result["trained"] = True
        dataset_result["history"] = history

        export_path = export_softmax_predictions(
            model=models_by_dataset[dataset_name],
            dataset_name=dataset_name,
            split=export_split,
            max_batches=max_export_batches,
        )
        dataset_result["exported"] = True
        dataset_result["export_path"] = str(export_path)
        results[dataset_name] = dataset_result

    return results


# Default run is preflight-only (safe with partial staging).
pipeline_results = run_full_pipeline_all_datasets(run_training=False)
pipeline_results


=== octmnist ===
preflight train availability: 64/64

=== pathmnist ===
preflight train availability: 64/64

=== tissuemnist ===
preflight train availability: 64/64


{'octmnist': {'preflight_found': 64,
  'preflight_checked': 64,
  'trained': False,
  'exported': False,
  'message': 'Preflight only. Set run_training=True to train/export.'},
 'pathmnist': {'preflight_found': 64,
  'preflight_checked': 64,
  'trained': False,
  'exported': False,
  'message': 'Preflight only. Set run_training=True to train/export.'},
 'tissuemnist': {'preflight_found': 64,
  'preflight_checked': 64,
  'trained': False,
  'exported': False,
  'message': 'Preflight only. Set run_training=True to train/export.'}}

In [ ]:
# pipeline_results = run_full_pipeline_all_datasets(
#     run_training=True,
#     epochs=20,
#     max_train_batches=30,
#     max_val_batches=5,
#     max_export_batches=5,
#     export_split="test",
# )


=== octmnist ===
preflight train availability: 64/64
[octmnist] preflight train image availability: 64/64
[octmnist] Epoch 1/20 | train batch 0: loss=1.3948 acc=0.1250
[octmnist] Epoch 1/20 | val batch 0: loss=1.1508 acc=0.4609
[octmnist] epoch 1/20 | train_loss=1.2209 train_acc=0.4596 | val_loss=1.1764 val_acc=0.4484
[octmnist] saved new best checkpoint -> /home/verdantliberatus/University/CMPUT469/repository/Snowdrop469Project/CNN/checkpoints/octmnist/best.pt
[octmnist] Epoch 2/20 | train batch 0: loss=1.0930 acc=0.5312
[octmnist] Epoch 2/20 | val batch 0: loss=1.1332 acc=0.4766
[octmnist] epoch 2/20 | train_loss=1.1792 train_acc=0.4672 | val_loss=1.1648 val_acc=0.4578
[octmnist] saved new best checkpoint -> /home/verdantliberatus/University/CMPUT469/repository/Snowdrop469Project/CNN/checkpoints/octmnist/best.pt
[octmnist] Epoch 3/20 | train batch 0: loss=1.1371 acc=0.4609
[octmnist] Epoch 3/20 | val batch 0: loss=1.0691 acc=0.5938
[octmnist] epoch 3/20 | train_loss=1.1425 train_acc